In [2]:
# 1.) dékodování vocabu
# Zjištění kódování
common_encodings = [
    "utf-8", "windows-1250", "latin1", "iso-8859-2", "cp1252"
]

for encoding in common_encodings:
    try:
        with open("vocab", "r", encoding=encoding) as file:
            content = file.read()
            print(f"Successfully decoded with encoding: {encoding}")
            print(content[:200])
            break
    except UnicodeDecodeError:
        print(f"Failed with encoding: {encoding}")

input_file = "vocab"
output_file = "vocab_utf8.txt"

# Otevření a přečtení původního souboru
with open(input_file, "r", encoding="windows-1250") as file:
    content = file.read()

# Uložení v novém kódování
with open(output_file, "w", encoding="utf-8") as file:
    file.write(content)

print(f"Soubor byl úspěšně převeden do UTF-8 a uložen jako '{output_file}'.")    

Failed with encoding: utf-8
Successfully decoded with encoding: windows-1250
a		a
aby		a b i
akcie		a k c i j e
akcii		a k c i j i
akcionář		a k c i j o n A Q
akciového		a k c i j o v E h o
akciových		a k c i j o v I x
akcií		a k c i j I
akt		a k t
akt		a g d
ale		a l e
ani		a
Soubor byl úspěšně převeden do UTF-8 a uložen jako 'vocab_utf8.txt'.


In [7]:
# 2.) Vytvoření listu listů - seznam seznamů + další s transkripcí dohromady
input_file = "vocab_utf8.txt"

list_listu = []                             # Základní list listů
list_listu = [['#', '#']]                   # Přidání na začátek seznamu '#'
list_with_transcriptions = []               # Nový seznam pro slovo a celou fonetickou transkripci dohromady
list_with_transcriptions = [['#', '#']]     # Přidání na začátek seznamu '#'

with open(input_file, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue
        
        # Rozdělení na slovo a fonémy
        casti = line.split("\t")
        if len(casti) != 2:
            casti = line.split(maxsplit=1)
        
        # Rozdělení fonémů podle mezer
        if len(casti) == 2:
            slovo = casti[0]
            fonemss = casti[1].split()
            list_listu.append([slovo] + fonemss)

            # Vytvoření fonetické transkripce jako jednoho řetězce
            transcription = "".join(fonemss)
            list_with_transcriptions.append([slovo, transcription])

# Výstup pro kontrolu - list listů
print("List listu (prvni 3):")
for entry in list_listu[:3]:
    print(entry)
print("--------------------------------------")

# Výstup pro kontrolu - celá fonetická transkripce
print("List listu s celou transkripci (prvni 3):")
for entry in list_with_transcriptions[:3]:
    print(entry)
print("--------------------------------------")


# 3.) Načtení potřebných souborů - leaves, 00010026_02 (pozorování)
import numpy as np

# A.) Načtení souboru leaves.txt
leaves_file = "leaves.txt"
fonemy = []
a_self = []     # Pravděpodobnosti setrvání ve stejném fonému
a_other = []    # Pravděpodobnosti přechodu do jiného fonému

with open(leaves_file, "r", encoding="utf-8") as file:
    for line in file:
        parts = line.strip().split()
        fonemy.append(parts[0])              # Foném
        a_self.append(float(parts[1]))       # Pravděpodobnost setrvání
        a_other.append(float(parts[2]))      # Pravděpodobnost přechodu

a_self_map = dict(zip(fonemy, a_self))
a_other_map = dict(zip(fonemy, a_other))

# B.) Načtení souboru s vektory pozorování
pozorov_file = "00010026_02.txt"
pozorovani = np.loadtxt(pozorov_file)


# 4.) Inicializace Viterbiho algoritmu
b_o1 = pozorovani[0]

# A.) Výpočet Phi_j(t=1)
Phi_t1 = []
for j in range(len(fonemy)):
    b_j_o1 = b_o1[j]
    Phi_t1.append(-np.log10(b_j_o1))

# B. Kombinování list_listu a Phi_t1 pro t=1
list_listu_s_Phi1 = []

# Pro každý vstup v list_listu se přidá odpovídající hodnota Phi_j(t=1)
for word_fonemes in list_listu:
    slovo = word_fonemes[0]
    fonemy_ve_slove = word_fonemes[1:]
    slovo_s_Phi = [slovo]
    
    # Pro první foném se přidá odpovídající Phi_j(t=1) hodnota
    if len(fonemy_ve_slove) > 0:
        first_fonem = fonemy_ve_slove[0]
        index = fonemy.index(first_fonem)
        slovo_s_Phi.append(Phi_t1[index])
    
    # Pro ostatní fonémy se nastaví hodnota inf
    slovo_s_Phi.extend([float('inf')] * (len(fonemy_ve_slove) - 1))
    
    list_listu_s_Phi1.append(slovo_s_Phi)

# C. Výstup
print("List listu s Phi hodnotami pro t=0/1 (prvni 3):")
for entry in list_listu_s_Phi1[:3]:
    print(entry)

# Zjištění dimenze výstupu
print("Počet řádků (slov) v list_listu_s_Phi1:", len(list_listu_s_Phi1))


# 5.) Funkce na přesun mezi koncem slova do začátku jiného - hledání minima
def find_min(list_listu_s_Phi, a_other_map, list_with_transcriptions):
    mins = []
    
    for i in range(len(list_listu_s_Phi)):
        hodnota_phi = list_listu_s_Phi[i][1:]               # Hodnoty phi (druhý prvek)
        fon_transkrip = list_with_transcriptions[i][1]      # Fonetická transkripce (druhý prvek)
        
        for j in range(len(hodnota_phi) - 1, -1, -1):       # Iterace od konce seznamu s phi hodnotami
            if hodnota_phi[j] != float('inf'):
                fonemek = fon_transkrip[j]
                ppst = hodnota_phi[j] - np.log10(a_other_map[fonemek])
                mins.append(ppst)
                break
                
    return min(mins)                                        # Vrátíme nejnižší pravděpodobnost

# Volání fce - pro ověření, po inicializaci Viterbiho
min_value = find_min(list_listu_s_Phi1, a_other_map, list_with_transcriptions)
print("Minimum pro t=0/1:", min_value)
print("--------------------------------------")


# 6.) Iterativní výpočet Viterbiho pro t = 1/2, ..., 804/805
T = len(pozorovani)                             # Počet časových okamžiků
list_listu_s_Phi = list_listu_s_Phi1.copy()     # Inicializace Viterbiho

for t in range(1, T):
    b_ot = pozorovani[t]

    # A. Najdi minimální hodnotu přechodu mezi koncem slova a začátkem jiného slova
    end_min = find_min(list_listu_s_Phi, a_other_map, list_with_transcriptions)

    # B. Viterbiho výpočet
    updated_list_listu_s_Phi = []
    processed_words = {}

    for entry in list_listu_s_Phi:        
        slovo = entry[0]
        phi = entry[1:]

        # C. Zjištění, která transkripce je pro toto slovo aktuálně potřebná
        if slovo not in processed_words:
            processed_words[slovo] = 0      # Stejné slovo poprvé - index
        else:
            processed_words[slovo] += 1     # Jiná fon. transkripce pro stejné slovo - index
        
        # D. Vybereme fonetickou transkripci na základě aktuálního indexu
        phonemes = [item[1:] for item in list_listu if item[0] == slovo]
        if processed_words[slovo] < len(phonemes):
            phonemes = phonemes[processed_words[slovo]]
        else:
            phonemes = phonemes[0]

        new_phi = phi.copy()

        for j in range(len(phonemes)):
            current_phoneme = phonemes[j]
            index = fonemy.index(current_phoneme)   # Index fonému
            b_j_ot = b_ot[index]                    # Vektory pozorání pro aktuální foném

            if j == 0:
                min1 = end_min - np.log10(b_j_ot)       # Přechod z konce jiného slova na začátek aktuálního
                min2 = phi[j] - np.log10(a_self_map[current_phoneme]) - np.log10(b_j_ot)    # Setrvání na začátku slova
                new_phi[j] = min(min1, min2)
            else:
                # Přechod mezi fonémy uvnitř slova
                prev_phoneme = phonemes[j - 1]
                min1 = phi[j - 1] - np.log10(a_other_map[prev_phoneme])     # Přechod z předchozího fonému
                min2 = phi[j] - np.log10(a_self_map[current_phoneme])       # Setrvání na aktuálním fonému
                new_phi[j] = min(min1, min2) - np.log10(b_j_ot)

        updated_list_listu_s_Phi.append([slovo] + new_phi)      # Aktualizovaný phi pro aktuální slovo

    list_listu_s_Phi = updated_list_listu_s_Phi     # Nahrazení starého listu novými hodnotami - pro danou iteraci

# Výsledný stav po posledním časovém okamžiku
print(f"Výsledný stav list_listu_s_Phi pro t={t}:")
for entry in list_listu_s_Phi[:3]:
    print(entry)

min_value = find_min(list_listu_s_Phi, a_other_map, list_with_transcriptions)
print(f"Hodnota minima pro poslední časový okamžik (pro t={t}):", min_value)
print("--------------------------------------")


List listu (prvni 3):
['#', '#']
['a', 'a']
['aby', 'a', 'b', 'i']
--------------------------------------
List listu s celou transkripci (prvni 3):
['#', '#']
['a', 'a']
['aby', 'abi']
--------------------------------------
List listu s Phi hodnotami pro t=0/1 (prvni 3):
['#', 0.0003273642420868833]
['a', 11.314470445929834]
['aby', 11.314470445929834, inf, inf]
Počet řádků (slov) v list_listu_s_Phi1: 916
Minimum pro t=0/1: 1.4443156968622255
--------------------------------------
Výsledný stav list_listu_s_Phi pro t=804:
['#', 575.5398510073536]
['a', 582.4148494758915]
['aby', 582.4148494758915, 588.5767503636057, 594.6494015604598]
Hodnota minima pro poslední časový okamžik (pro t=804): 576.9838393399738
--------------------------------------


In [ ]:
# 7.) Načtení unigramů z jazykového modelu + konstanty pro dál
cesta_k_jaz_modelu = "jazykovy_model.arpa"
unigrams = {}
with open(cesta_k_jaz_modelu, 'r', encoding='utf-8') as file:
    lines = file.readlines()

# začátek unigramů
start_index = lines.index('\\1-grams:\n') + 1

# Unigramy
for line in lines[start_index:]:
    line = line.strip()
    if line.startswith('\\') or line == '':
        break

    # V řádku unigram a ppst.
    parts = line.split()
    if len(parts) >= 2:
        probability = float(parts[0])
        unigram = parts[1]
        unigrams[unigram] = probability

print(unigrams)

L_gamma = -np.log10(0.01)     # penalta vložení slova 
beta = 5                      # váha jazykového modelu


{'valná': -2.7090363271761175, 'hromada': -2.7153765052071366, 'severočeské': -2.7283414823715044, 'plynárenské': -2.7153765052071366, 'rozhodla': -2.472338456520842, 'že': -2.079538087501216, 'vyplatí': -2.7153765052071366, 'dividendy': -2.7090363271761175, 'pouze': -2.7153765052071366, 'na': -1.9308850767924741, 'zaměstnanecké': -2.7485448684597915, 'akcie': -2.2999121515515664, '</s>': -1.1768175556416367, 'nejvýznamnější': -2.7920105622408817, 'pohyby': -2.777033406271229, 'hodnoty': -2.7697341675297293, 'koruny': -2.777033406271229, 'za': -2.072214229588943, 'posledních': -2.7697341675297293, 'šest': -2.4986673952431913, 'měsíců': -2.784457424350436, 'byly': -2.506492732755148, 'jednoznačně': -2.777033406271229, 'odrazem': -2.807522728419129, 'toku': -2.7920105622408817, 'krátkodobého': -2.784457424350436, 'kapitálu': -2.7697341675297293, 'v': -1.9256713949470778, 'nových': -4.246855422249392, '<unk>': -1.4140272927100384, 'období': -2.807522728419129, 'platném': -2.79201056224088